# Phase 4B-3 — MU-Glioma External Morphology Tests

Runs the same T1/T5/T8/T4/N9/N10/N11/N12 tests from Phase 4A on MU-Glioma-Post.
This validates that TaViT embeddings capture the same morphological/temporal properties
on an **entirely unseen external dataset**.

| Test | Measures | Threshold |
|---|---|---|
| T1 | Spearman(trajectory PC1, volume change) | ≥0.30 |
| T5 | Coherence — consecutive scan clustering | rho>0 |
| T8 | Kendall tau — monotone progression ranking | ≥0.30 |
| T4 | RANO AUC — detect ≥25% volume growth | ≥0.65 |
| N9 | Ratio head Spearman on GT log-ratio | ≥0.30 |
| N10 | 3-class F1 (percentile thresholds) | ≥0.40 |
| N11 | Silhouette — trajectory cluster quality | ≥0.10 |
| N12 | Cohen d — progressive vs stable separation | ≥0.50 |


In [ ]:
import subprocess, sys
subprocess.run([sys.executable,"-m","pip","install","-q","--no-deps","monai"], capture_output=True)
subprocess.run([sys.executable,"-m","pip","install","-q","scipy","scikit-learn","nibabel"], capture_output=True)

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import warnings, json
from pathlib import Path
from scipy.stats import spearmanr, kendalltau
from scipy.stats import mannwhitneyu
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, roc_auc_score, f1_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.linear_model import LogisticRegression
from collections import defaultdict
warnings.filterwarnings("ignore")
print("Imports OK")


In [ ]:
# ============================================================
# LOAD ALL DATA FROM B1 + B2 OUTPUTS
# ============================================================
SEARCH = [Path("/kaggle/input"), Path("/kaggle/working")]

def find(names):
    for root in SEARCH:
        if not root.exists(): continue
        for f in root.rglob("*"):
            if any(n in f.name for n in names): return f
    return None

# TaViT trajectories: {pid: (256,)} per-patient dict
traj_path = find(["mu_glioma_trajectories.npz"])
assert traj_path, "mu_glioma_trajectories.npz not found"
traj_npz = np.load(traj_path, allow_pickle=True)
traj_pids = list(traj_npz.files)
TRAJ_DIM = traj_npz[traj_pids[0]].shape[0]
print(f"TaViT trajectories: {len(traj_pids)} patients | dim={TRAJ_DIM}")

# Static hybrid embeddings: (N, 4617) with patient_ids and timepoints
emb_path = find(["mu_glioma_embeddings.npz"])
assert emb_path, "mu_glioma_embeddings.npz not found"
emb_data = np.load(emb_path, allow_pickle=True)
emb_arr  = emb_data["embeddings"]       # (N, 4617)
emb_pids = emb_data["patient_ids"]      # (N,)
emb_tps  = emb_data["timepoints"]       # (N,)
print(f"Static embeddings: {emb_arr.shape}")

# GT volumes: gt_wt/gt_tc/gt_et in mm3 per scan (from B2)
gt_vol_path = find(["gt_volumes.csv"])
if gt_vol_path:
    gt_df = pd.read_csv(gt_vol_path)
    gt_df = gt_df.dropna(subset=["gt_wt"])
    print(f"GT volumes: {len(gt_df)} scans ({gt_df['patient_id'].nunique()} patients)")
else:
    raise FileNotFoundError("gt_volumes.csv not found — run B2 first")

# Inference results: predicted_ratio per patient (from B1)
res_path = find(["mu_glioma_results.csv"])
assert res_path, "mu_glioma_results.csv not found"
results_df = pd.read_csv(res_path)
print(f"Inference results: {len(results_df)} patients")

# Ratio head + scaler + isotonic calibrator (from trained artifacts)
rh_path  = find(["ratio_head.pt"])
sc_path  = find(["ratio_head_scaler.pkl"])
iso_path = find(["ratio_head_iso_cal.pkl"])
assert all([rh_path, sc_path, iso_path]), "ratio_head artifacts not found"
print(f"Ratio head artifacts found")


In [ ]:
# ============================================================
# BUILD PATIENT LABELS FROM GT MASK VOLUMES
# delta_vol, v_first, v_last, log_ratio, response class
# ============================================================

patient_labels = {}
for pid in gt_df["patient_id"].unique():
    pid_rows = gt_df[gt_df["patient_id"]==pid].sort_values("timepoint")
    if len(pid_rows) < 2: continue
    v_first = float(pid_rows.iloc[0]["gt_wt"])
    v_last  = float(pid_rows.iloc[-1]["gt_wt"])
    if v_first < 1: continue
    delta_vol = v_last - v_first
    gt_ratio  = delta_vol / v_first
    log_ratio = float(np.log(max(v_last,1.0) / max(v_first,1.0)))
    if   gt_ratio >=  0.25: response = "progressive"
    elif gt_ratio <= -0.25: response = "responder"
    else:                   response = "stable"
    patient_labels[pid] = {
        "v_first": v_first, "v_last": v_last,
        "delta_vol": delta_vol, "gt_ratio": gt_ratio,
        "log_ratio": log_ratio, "response": response
    }

print(f"Patients with >=2 GT timepoints: {len(patient_labels)}")
cls_dist = pd.Series([v["response"] for v in patient_labels.values()]).value_counts()
print(f"GT class distribution: {cls_dist.to_dict()}")

# Overlap: patients with BOTH trajectory and labels
eval_pids = [p for p in traj_pids if p in patient_labels]
print(f"Evaluation set (traj + GT labels): {len(eval_pids)} patients")

X_traj   = np.stack([traj_npz[p].astype(np.float32) for p in eval_pids])   # (N, 256)
y_delta  = np.array([patient_labels[p]["delta_vol"] for p in eval_pids])
y_ratio  = np.array([patient_labels[p]["gt_ratio"]  for p in eval_pids])
y_log    = np.array([patient_labels[p]["log_ratio"]  for p in eval_pids])
y_class  = np.array([patient_labels[p]["response"]   for p in eval_pids])
v_firsts = np.array([patient_labels[p]["v_first"]     for p in eval_pids])

X_scaled = StandardScaler().fit_transform(X_traj)
print(f"X_traj: {X_traj.shape} | classes: {dict(zip(*np.unique(y_class,return_counts=True)))}")


In [ ]:
# ============================================================
# TEMPORAL TESTS: T1 | T5 | T8 | T4
# ============================================================
results = {}

# -- PCA decomposition of trajectory space --
pca = PCA(n_components=min(10, X_scaled.shape[0]-1, X_scaled.shape[1]))
X_pc = pca.fit_transform(X_scaled)   # (N, n_pc)

print("="*60)
print("  TEMPORAL TESTS (MU-Glioma External, {len(eval_pids)} patients)")
print("="*60)

# --- T1: Spearman(PC1, |delta_vol / v_first|) ---
abs_ratio = np.abs(y_ratio)
rho_t1_pc1, p_t1_pc1 = spearmanr(X_pc[:, 0], abs_ratio)
rho_t1_norm, p_t1_norm = spearmanr(np.linalg.norm(X_traj, axis=1), abs_ratio)
# Try all PCs, report best
best_rho_t1 = max([abs(spearmanr(X_pc[:, k], abs_ratio)[0]) for k in range(X_pc.shape[1])])
print(f"\n  T1 — Spearman(trajectory, |volume change ratio|)")
print(f"    PC1      : rho={rho_t1_pc1:.3f}  p={p_t1_pc1:.2e}")
print(f"    ||norm|| : rho={rho_t1_norm:.3f}  p={p_t1_norm:.2e}")
print(f"    Best PC  : |rho|={best_rho_t1:.3f}")
t1_rho = rho_t1_pc1
results["T1"] = {"rho": t1_rho, "p": p_t1_pc1, "pass": abs(t1_rho)>=0.30 and p_t1_pc1<0.05}
print(f"    -> {'PASS' if results['T1']['pass'] else 'BELOW'} (|rho|>={0.30})")

# --- T5: Coherence — consecutive scans cluster closer than random ---
# Use static 4617-D embeddings per scan grouped by patient
scan_by_pid = defaultdict(list)
for k, (pid, tp) in enumerate(zip(emb_pids, emb_tps)):
    scan_by_pid[str(pid)].append((int(tp), emb_arr[k]))

consecutive_dists, random_dists = [], []
rng = np.random.default_rng(42)
all_pids_list = list(scan_by_pid.keys())
for pid, scans in scan_by_pid.items():
    if len(scans) < 2: continue
    scans_sorted = sorted(scans, key=lambda x: x[0])
    for i in range(len(scans_sorted)-1):
        e1 = scans_sorted[i][1]
        e2 = scans_sorted[i+1][1]
        consecutive_dists.append(np.linalg.norm(e1 - e2))
    # Random: pick scan from different patient
    other_pid = rng.choice([p for p in all_pids_list if p != pid])
    rand_scan = scan_by_pid[other_pid][0][1]
    random_dists.append(np.linalg.norm(scans_sorted[0][1] - rand_scan))

rho_t5, p_t5 = spearmanr(
    [0]*len(consecutive_dists) + [1]*len(random_dists),
    consecutive_dists + random_dists
)
mean_consec = np.mean(consecutive_dists)
mean_rand   = np.mean(random_dists)
results["T5"] = {"consec": mean_consec, "random": mean_rand, "pass": mean_consec < mean_rand}
print(f"\n  T5 — Coherence (consecutive vs random scan distance)")
print(f"    Consecutive dist: {mean_consec:.2f}")
print(f"    Random dist:      {mean_rand:.2f}")
print(f"    -> {'PASS (consecutive < random)' if results['T5']['pass'] else 'FAIL'}")

# --- T8: Kendall tau(PC1, signed delta_vol) ---
tau_t8, p_t8 = kendalltau(X_pc[:, 0], y_delta)
rho_t8, p_t8r = spearmanr(X_pc[:, 0], y_delta)
results["T8"] = {"tau": tau_t8, "p": p_t8, "pass": abs(tau_t8)>=0.20 and p_t8<0.05}
print(f"\n  T8 — Kendall tau(trajectory PC1, signed delta_vol_mm3)")
print(f"    tau={tau_t8:.3f}  p={p_t8:.2e}  (Spearman rho={rho_t8:.3f})")
print(f"    -> {'PASS' if results['T8']['pass'] else 'BELOW'} (|tau|>=0.20)")

# --- T4: RANO AUC — detect >=25% volume growth ---
# Use predicted_ratio from B1 results as score, GT RANO as binary label
res_sub = results_df[results_df["patient_id"].isin(eval_pids)][["patient_id","predicted_ratio"]].set_index("patient_id")
rano_scores, rano_labels = [], []
for pid in eval_pids:
    if pid not in res_sub.index: continue
    gt_prog = 1 if patient_labels[pid]["gt_ratio"] >= 0.25 else 0
    pred_score = float(res_sub.loc[pid, "predicted_ratio"])
    rano_scores.append(pred_score)
    rano_labels.append(gt_prog)

if len(set(rano_labels)) == 2 and len(rano_labels) >= 10:
    auc_t4 = roc_auc_score(rano_labels, rano_scores)
    results["T4"] = {"auc": auc_t4, "pass": auc_t4>=0.60}
    print(f"\n  T4 — RANO AUC (predicted_ratio vs GT >=25% growth, n={len(rano_labels)})")
    print(f"    AUC={auc_t4:.3f}  -> {'PASS' if results['T4']['pass'] else 'BELOW'} (>=0.60)")
else:
    results["T4"] = {"auc": float("nan"), "pass": False}
    print(f"\n  T4 — RANO AUC: insufficient samples")


In [ ]:
# ============================================================
# TRAJECTORY-SPECIFIC TESTS: N9 | N10 | N11 | N12
# ============================================================
import pickle

# Load ratio head
class RatioHead(nn.Module):
    def __init__(self, d=256):
        super().__init__()
        self.head = nn.Linear(d, 1)
    def forward(self, x): return self.head(x).squeeze(-1)

ratio_head = RatioHead(TRAJ_DIM)
ratio_head.load_state_dict(torch.load(str(rh_path), map_location="cpu"))
ratio_head.eval()

with open(str(sc_path), "rb") as f:  scaler_rh = pickle.load(f)
with open(str(iso_path), "rb") as f: iso_cal   = pickle.load(f)

# Apply ratio head to MU-Glioma trajectories
X_rh_scaled = torch.tensor(scaler_rh.transform(X_traj), dtype=torch.float32)
with torch.no_grad():
    raw_scores = ratio_head(X_rh_scaled).numpy()   # uncalibrated log-ratio predictions
    cal_scores = iso_cal.predict(raw_scores)        # isotonic calibrated

# --- N9: Spearman(ratio_head output, GT log-ratio) ---
rho_n9_raw, p_n9_raw = spearmanr(raw_scores, y_log)
rho_n9_cal, p_n9_cal = spearmanr(cal_scores, y_log)
rho_n9_delta, p_n9_d = spearmanr(raw_scores, y_ratio)  # vs relative ratio (not log)
results["N9"] = {"rho_raw": rho_n9_raw, "rho_cal": rho_n9_cal, "p": p_n9_raw,
                 "pass": abs(rho_n9_raw)>=0.25 and p_n9_raw<0.05}
print("="*60)
print(f"  TRAJECTORY TESTS (n={len(eval_pids)})")
print("="*60)
print(f"\n  N9 — Ratio head Spearman (MU-Glioma GT log-ratio)")
print(f"    Raw  : rho={rho_n9_raw:.3f}  p={p_n9_raw:.2e}")
print(f"    Cal  : rho={rho_n9_cal:.3f}  p={p_n9_cal:.2e}")
print(f"    vs relative ratio: rho={rho_n9_delta:.3f}  p={p_n9_d:.2e}")
print(f"    -> {'PASS' if results['N9']['pass'] else 'BELOW'} (|rho|>=0.25, p<0.05)")

# --- N10: 3-class F1 using percentile thresholds ---
p_prog = np.percentile(raw_scores, 51)
p_resp = np.percentile(raw_scores, 30)
def cls_pct(s):
    if s >= p_prog: return "progressive"
    if s <= p_resp: return "responder"
    return "stable"
y_pred_cls = np.array([cls_pct(s) for s in raw_scores])
labels_order = ["progressive","stable","responder"]
f1_macro = f1_score(y_class, y_pred_cls, labels=labels_order, average="macro", zero_division=0)
f1_weighted = f1_score(y_class, y_pred_cls, labels=labels_order, average="weighted", zero_division=0)
results["N10"] = {"f1_macro": f1_macro, "f1_weighted": f1_weighted, "pass": f1_macro>=0.35}
print(f"\n  N10 — 3-class F1 (percentile p30/p51 thresholds)")
print(f"    F1 macro={f1_macro:.3f}  weighted={f1_weighted:.3f}")
print(f"    GT  dist: {dict(zip(*np.unique(y_class,return_counts=True)))}")
print(f"    Pred dist: {dict(zip(*np.unique(y_pred_cls,return_counts=True)))}")
print(f"    -> {'PASS' if results['N10']['pass'] else 'BELOW'} (macro>=0.35)")

# --- N11: Silhouette score on trajectory embeddings ---
sil_scores = {}
for k in [2, 3]:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels_km = km.fit_predict(X_scaled)
    if len(np.unique(labels_km)) >= 2:
        sil_scores[k] = silhouette_score(X_scaled, labels_km)
results["N11"] = {"k2": sil_scores.get(2,0), "k3": sil_scores.get(3,0),
                  "pass": max(sil_scores.values(), default=0) >= 0.10}
print(f"\n  N11 — Silhouette (trajectory clustering quality)")
for k, s in sil_scores.items():
    print(f"    K={k}: Sil={s:.3f}")
print(f"    -> {'PASS' if results['N11']['pass'] else 'BELOW'} (max Sil >= 0.10)")

# --- N12: Cohen's d (progressive vs stable in trajectory space) ---
prog_mask  = y_class == "progressive"
stable_mask = y_class == "stable"
n_prog, n_stab = prog_mask.sum(), stable_mask.sum()
if n_prog >= 5 and n_stab >= 5:
    pc1_prog  = X_pc[prog_mask,  0]
    pc1_stab  = X_pc[stable_mask, 0]
    pooled_sd = np.sqrt(((n_prog-1)*pc1_prog.std()**2 + (n_stab-1)*pc1_stab.std()**2)
                        / (n_prog + n_stab - 2))
    cohens_d  = abs(pc1_prog.mean() - pc1_stab.mean()) / (pooled_sd + 1e-9)
    stat, p_n12 = mannwhitneyu(pc1_prog, pc1_stab, alternative="two-sided")
    results["N12"] = {"cohens_d": cohens_d, "p": p_n12, "pass": cohens_d>=0.40}
    print(f"\n  N12 — Cohen d (progressive vs stable, trajectory PC1)")
    print(f"    n_prog={n_prog}  n_stable={n_stab}")
    print(f"    Cohen d={cohens_d:.3f}  p={p_n12:.3e}")
    print(f"    -> {'PASS' if results['N12']['pass'] else 'BELOW'} (d>=0.40)")
else:
    results["N12"] = {"cohens_d": 0, "p": 1, "pass": False}
    print(f"\n  N12: insufficient class samples (prog={n_prog}, stable={n_stab})")


In [ ]:
# ============================================================
# FINAL REPORT
# ============================================================
print("\n" + "="*65)
print("  MU-GLIOMA-POST — MORPHOLOGY TEST REPORT (Phase 4B-3)")
print(f"  n={len(eval_pids)} patients  |  External validation (zero training exposure)")
print("="*65)

rows = [
    ("T1", "Spearman(traj PC1, |vol change|)",  f"rho={results['T1']['rho']:.3f}",   ">=0.30"),
    ("T5", "Coherence (consec < rand dist)",     f"{results['T5']['consec']:.1f} < {results['T5']['random']:.1f}", "consec<rand"),
    ("T8", "Kendall tau(PC1, delta_vol)",        f"tau={results['T8']['tau']:.3f}",    ">=0.20"),
    ("T4", "RANO AUC (>=25% detection)",         f"AUC={results.get('T4',{}).get('auc','N/A'):.3f}" if isinstance(results.get('T4',{}).get('auc'),float) else "N/A", ">=0.60"),
    ("N9", "Ratio head Spearman (GT log-ratio)", f"rho={results['N9']['rho_raw']:.3f}", ">=0.25"),
    ("N10","3-class F1 (percentile)",            f"F1={results['N10']['f1_macro']:.3f}", ">=0.35"),
    ("N11","Silhouette K=2",                     f"Sil={results['N11'].get('k2',0):.3f}", ">=0.10"),
    ("N12","Cohen d (prog vs stable)",           f"d={results['N12']['cohens_d']:.3f}",  ">=0.40"),
]
passed = sum(1 for k in ["T1","T5","T8","T4","N9","N10","N11","N12"] if results.get(k,{}).get("pass",False))
total  = 8

print(f"\n  {'Test':<5} {'Metric':<38} {'Value':<20} {'Threshold':<12} {'Verdict'}")
print(f"  {'-'*5} {'-'*38} {'-'*20} {'-'*12} {'-'*7}")
test_keys = ["T1","T5","T8","T4","N9","N10","N11","N12"]
for (k, desc, val, thr), tk in zip(rows, test_keys):
    p = results.get(tk,{}).get("pass",False)
    print(f"  {k:<5} {desc:<38} {val:<20} {thr:<12} {'PASS' if p else 'BELOW'}")

print(f"\n  Overall: {passed}/{total} PASS")
print("="*65)

# Compare with BraTS Phase 4A results
print("\n  BraTS (Phase 4A) vs MU-Glioma (Phase 4B-3) comparison:")
print(f"  {'Test':<8} {'BraTS':<15} {'MU-Glioma'}")
print(f"  {'-'*8} {'-'*15} {'-'*15}")
brats = {"T1":"rho=0.608","T8":"tau=0.439","T4":"AUC=0.819","N9":"rho=0.569","N10":"F1=0.718","N11":"Sil=0.391","N12":"d=0.783"}
muglioma = {
    "T1": f"rho={results['T1']['rho']:.3f}",
    "T8": f"tau={results['T8']['tau']:.3f}",
    "T4": f"AUC={results.get('T4',{}).get('auc',float('nan')):.3f}",
    "N9": f"rho={results['N9']['rho_raw']:.3f}",
    "N10":f"F1={results['N10']['f1_macro']:.3f}",
    "N11":f"Sil={results['N11'].get('k2',0):.3f}",
    "N12":f"d={results['N12']['cohens_d']:.3f}",
}
for t in ["T1","T8","T4","N9","N10","N11","N12"]:
    print(f"  {t:<8} {brats.get(t,'N/A'):<15} {muglioma.get(t,'N/A')}")
